# Name That LLM

**Goal:** given one passage of text, guess which of 18 LLMs wrote it.

**The catch:** the training passages come from about 150 prompts, and every model answered every prompt once.
The test set uses *new* prompts, so the topic can't help us. Only writing style can.

This notebook runs **the same pipeline as the Airflow DAG** (`dags/name_that_llm_dag.py`), step by step, using the
same code (the `name_that_llm` package). Each section below is one Airflow task:

| Section | Airflow task | What it does |
|---|---|---|
| 1 | `check_data` | make sure `train.csv` is there |
| 2 | `find_prompt_groups` | cluster passages that answer the same prompt |
| 3 | `tune_with_optuna` | search for good settings, scored on prompts the model has not seen |
| 4 | `train_model` | train on everything and save the model |
| 5 | `evaluate_model` | the honest score (5-fold, whole prompts held out) |
| 6 | `register_model` | add a new version to the MLflow registry, promote it if it beats the champion |

Running it here trains and registers a model in a local MLflow store (`mlflow.db`). It does not touch the Docker
registry unless you set `MLFLOW_TRACKING_URI` to that server first.

In [ ]:
import os
import sys
from pathlib import Path

# make the package importable when the notebook is run from the project folder or from notebooks/
for folder in (Path.cwd(), Path.cwd().parent):
    if (folder / "src" / "name_that_llm").exists():
        sys.path.insert(0, str(folder / "src"))

import joblib
import numpy as np
import pandas as pd

from name_that_llm import ROOT
from name_that_llm.features import embed_passages, make_skeleton
from name_that_llm.grouping import cluster_passages, cluster_quality
from name_that_llm.predict import predict_models
from name_that_llm.registry import load_champion, register_from_files
from name_that_llm.train import cross_validate, fit_bundle, load_params, save_metrics, summarize_scores
from name_that_llm.tune import nested_cv, tune_and_save

TRIALS = int(os.environ.get("TRIALS", 15))   # how many Optuna trials to run (the DAG has the same setting)
RUN_NESTED_CHECK = False                      # True adds a 15-20 minute honest check of the tuning (section 3)

DATA = ROOT / "data"
ARTIFACTS = ROOT / "artifacts"

# where MLflow keeps runs and model versions; set this beforehand to use a shared server instead
os.environ.setdefault("MLFLOW_TRACKING_URI", f"sqlite:///{ROOT / 'mlflow.db'}")
print("project folder:", ROOT)
print("MLflow store  :", os.environ["MLFLOW_TRACKING_URI"])

## 1. Check the data  *(task `check_data`)*

In [ ]:
train_file = DATA / "train.csv"
assert train_file.exists(), f"train.csv is missing: put it in {DATA}"

df = pd.read_csv(train_file).dropna(subset=["text", "model"]).reset_index(drop=True)

n_models = df["model"].nunique()
print(len(df), "passages,", n_models, "models, about", round(len(df) / n_models), "prompts")

## 2. Find the hidden prompt groups  *(task `find_prompt_groups`)*

`train.csv` has a `prompt_id` column, but the test set won't. So we rebuild the groups ourselves:
turn each passage into topic numbers (TF-IDF on rare content words, squashed to 200 numbers with LSA), then cluster.
Only rare topical words are used, so passages cluster by *subject*, not by author.

**Did it work?** A good group has about 18 passages and never the same model twice. Since this file has the real
`prompt_id`, we can also compare with the truth (the adjusted Rand index, 1.0 = perfect).

In [ ]:
df["group"] = cluster_passages(df["text"], n_models=n_models)

quality = cluster_quality(df["group"], df["model"], df["prompt_id"])
print("groups:", quality["n_groups"])
print("median group size:", quality["median_size"])
print("groups with a repeated model: %.1f%%" % (100 * quality["share_with_repeat"]))
print("agreement with the real prompts:", round(quality["ari"], 3))

df.to_csv(DATA / "train_with_groups.csv", index=False)
groups = df["group"].to_numpy()

### Why the groups matter

They are used in three ways, and all three show up in the sections below:

* **Honest scoring.** If we split passages at random, the other 17 answers to the same prompt end up in training
  with *other* labels, which confuses the model. Splitting by prompt copies the real test situation.
* **Centering.** All answers in a group share the topic. Subtracting the group's average from each passage's numbers
  cancels the topic and leaves what is different about each author.
* **One model each.** Each model answers each prompt once, so inside a group the 18 passages should get 18 *different*
  models. We find the best one-to-one matching from the classifier's probabilities (the Hungarian algorithm).

### The style features

Each passage becomes TF-IDF numbers in three views: characters (2 to 5 letters in a row: spelling and punctuation habits),
words (single words and pairs: phrasing), and a **skeleton**, which keeps the 150 most common words and the punctuation
and replaces every other word with `W`. The skeleton captures sentence rhythm and has almost no topic in it:

In [ ]:
common_words = {"the", "it", "was"}
make_skeleton("The cat sat; it was warm.", common_words)

## 3. Tune the settings with Optuna  *(task `tune_with_optuna`)*

Three settings are searched: how strongly the style classifier is regularised (`style_c`), the same for the DistilBERT
classifier (`embedding_c`), and how much DistilBERT counts in the blend (`embedding_weight`). The very first trial is
always the hand-picked settings, so we can see whether tuning really beats them. Every trial is scored on prompts held
out of training.

DistilBERT numbers (768 per passage) come from a saved file. If it is missing they are computed once, which needs
`sentence-transformers` and takes a few minutes on a CPU.

In [ ]:
embedding_file = DATA / "embeddings" / "distilbert.npy"
embeddings = embed_passages(df["text"], embedding_file)
print("embeddings:", embeddings.shape)

In [ ]:
tuning = tune_and_save(df["text"], df["model"], groups, embeddings, TRIALS, ARTIFACTS / "best_params.json")

print("hand-picked settings scored", round(tuning["cv_score_default"], 3))
print("best tuned settings scored ", round(tuning["cv_score_tuned"], 3), "(on the same folds, so a bit optimistic)")
tuning["best_params"]

**Is the gain real?** The scores above come from the same folds that picked the settings, so they flatter the tuned
version. The fair test tunes *inside* each fold and scores on prompts the search never saw. When we ran it
(6 trials), tuned and hand-picked settings both scored **0.520**: in four of five folds the search just kept the
hand-picked values. So treat 0.52 as the real level. Switch `RUN_NESTED_CHECK` on at the top to repeat the check.

In [ ]:
if RUN_NESTED_CHECK:
    honest = nested_cv(df["text"], df["model"], groups, embeddings, n_trials=6)
    print(honest.mean().round(3))

## 4. Train the model  *(task `train_model`)*

In [ ]:
params = load_params(ARTIFACTS / "best_params.json")
print("using settings:", params)

bundle = fit_bundle(df["text"], df["model"], groups, embeddings, **params)

joblib.dump(bundle, ARTIFACTS / "model.joblib")
print("saved", ARTIFACTS / "model.joblib")

## 5. Evaluate it  *(task `evaluate_model`)*

Five folds, each holding out whole prompts. Scores are balanced accuracy (every model counts equally; guessing at
random would give 1/18 = 0.056), after centering and the one-model-each assignment. The `style only` row is the
TF-IDF model alone, and `style + embeddings` adds the DistilBERT blend.

This is a local estimate on the training data, not a score from the Kaggle competition.

In [ ]:
scores = cross_validate(df["text"], df["model"], groups, embeddings, params=params)
summary = summarize_scores(scores)

save_metrics(summary, params, ARTIFACTS / "metrics.json")
summary

## 6. Register the model  *(task `register_model`)*

Every trained model becomes a numbered version in the MLflow registry, with a semantic version tag (`0.1.0`, `0.1.1` ...)
and its score. The alias **`champion`** marks the version that gets served. A new version becomes the champion only if
its score is at least as good as the current champion's; otherwise it is kept and tagged `not promoted`.

Open the MLflow page to see the runs and versions: `mlflow ui --backend-store-uri` followed by the MLflow store
printed at the top.

In [ ]:
result = register_from_files(ARTIFACTS / "model.joblib", ARTIFACTS / "metrics.json", ARTIFACTS / "best_params.json")

print("registered version", result["version"], "with semantic version", result["semver"], "and score", round(result["score"], 3))
print("now the champion" if result["promoted"] else f"not promoted: the champion scores {result['champion_score']:.3f}")

## Try the champion

Load whatever the registry calls the champion and predict a batch of passages, as the API does. Centering and
assignment need the other passages from the same prompts, so the passages are grouped first; a batch under 54 passages
is too small to group and falls back to the weaker plain classifier.

These 10 prompts were part of the training data, so the accuracy here only shows that the model loads and runs. The
honest number is the one in section 5.

In [ ]:
champion, info = load_champion()
print("loaded:", info)

ten_prompts = sorted(df["prompt_id"].unique())[:10]
rows = df.index[df["prompt_id"].isin(ten_prompts)]

prediction = predict_models(champion, df["text"][rows], embeddings=embeddings[rows])
accuracy = (np.array(prediction.models) == df["model"][rows].to_numpy()).mean()
print(len(rows), "passages,", prediction.mode, "mode, accuracy on training prompts:", round(accuracy, 3))

## What to take away

* Splitting by prompt gives the honest score. Random splits mislead.
* Centering and the one-model-each assignment both help a lot, because both use the fact that each prompt has exactly one
  answer per model. The skeleton features and a small share of DistilBERT add a further gain.
* The honest level is about **0.52 balanced accuracy** (chance is 0.056). Optuna tuning did not improve it once checked fairly.
* Models from the same company (for example the Gemini ones) get mixed up with each other most.
* At test time the same recipe needs the test passages grouped by prompt too.
* Every model is versioned in the registry, and only a model that is not worse than the champion replaces it.